# ДЗ 3. Дообучение BERT для распознавания именованных сущностей (NER)

---

### Зачем это нужно

Извлечение именованных сущностей (Named Entity Recognition, NER) —  людей, организаций, географических объектов и других сущностей —  это один из первых шагов превращения неструктурированного текста в структурированные данные. На этой задаче строится поиск по сущностям, аналитика, извлечение информации и базы знаний.

Мы разобрали, как предобученную модель BERT можно адаптировать к новой задаче, заменив выходную классификационную голову и дообучив модель на размеченных данных. В этом домашнем задании вы проделаете этот процесс самостоятельно: возьмете предобученный русскоязычный BERT и дообучите его для задачи разметки последовательностей.

Главный результат работы —  не достижение высоких значений метрики, а корректная и осмысленная оценка качества модели. Вы увидите, почему в NER оценки на уровне токенов могут вводить в заблуждение, научитесь интерпретировать метрики на уровне сущностей и анализировать типичные ошибки модели.

### Что вы сделаете

1. Загрузите русскоязычную часть датасета WikiANN и разберетесь с BIO-разметкой и дисбалансом класса `O` (блок 1).
2. Разберетесь, как BIO-метки слов сопоставляются с подсловными токенами BERT (блок 2).
3. Соберете модель AutoModelForTokenClassification и дообучите ее на задаче NER (блок 3).
4. Сравните метрики token-level и entity-level и объясните, почему они могут существенно различаться (блок 4).
5. Проанализируете ошибки модели: неверные границы сущностей, ошибки в определении типа и особенности, связанные с подсловной токенизацией (блок 5).
6. Сформулируете обоснованные выводы по результатам эксперимента (блок 6).

> **Сколько займет:** ориентировочно 4— 6 часов.

> **Про объем кода:** не превращайте ноутбук в полотно, пишите функциями, комментируйте код. Лаконичное и читаемое решение оценивается выше длинного и запутанного.

> **Требуется GPU.** В Google Colab выберите Runtime → Change runtime type → GPU. Без CPU обучение модели будет занимать слишком много времени.

## Настройка окружения

In [ ]:
!pip install -q -U transformers datasets seqeval accelerate

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import (
    classification_report as seq_classification_report,
    f1_score as seq_f1,
    precision_score as seq_precision,
    recall_score as seq_recall,
)
from sklearn.metrics import classification_report as sk_classification_report

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

MODEL = "cointegrated/rubert-tiny2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("GPU не найден, рекомендуется включить GPU.")

## 1. Данные и BIO-разметка (1 балл)

**Датасет:** [`WikiANN (ru)`](https://huggingface.co/datasets/unimelb-nlp/wikiann) размечен по трем типам сущностей —  `PER` (люди), `ORG` (организации), `LOC` (локации) —  в схеме BIO. Каждый пример —  это список слов (`tokens`) и список меток (`ner_tags`) той же длины. Ячейка ниже грузит датасет.

In [ ]:
raw = load_dataset("unimelb-nlp/wikiann", "ru")

# Берем подвыборку (полный train не нужен для целей ДЗ)
raw["train"] = raw["train"].shuffle(seed=RANDOM_STATE).select(range(8000))
raw["validation"] = raw["validation"].shuffle(seed=RANDOM_STATE).select(range(2000))
raw["test"] = raw["test"].shuffle(seed=RANDOM_STATE).select(range(2000))

# Имена меток: индекс = id метки
label_names = raw["train"].features["ner_tags"].feature.names
id2label = {i: t for i, t in enumerate(label_names)}
label2id = {t: i for i, t in enumerate(label_names)}

print("Метки:", label_names)
print("Размеры:", {k: len(v) for k, v in raw.items()})

# Как выглядит один пример:
example = raw["train"][1]
print("\nПример:")
for w, t in zip(example["tokens"], example["ner_tags"]):
  print(f"{w:<20} {id2label[t]}")

### ✍️ Задание 1.1. Распределение меток и длин предложений

По обучающей выборке `raw["train"]` выполните следующие действия:

1. Постройте столбчатую диаграмму распределения BIO-меток (количество токенов каждой метки).
2. Постройте гистограмму распределения длин предложений (в токенах).
3. Вычислите долю токенов с меткой `O` относительно общего числа токенов.

Требования: у каждого графика должны быть подписаны оси и указан заголовок.

>*Подсказка.* Все метки можно собрать в один список:
>```
>all_tags = [t for row in raw["train"]["ner_tags"] for t in row]
>```
>После этого преобразуйте идентификаторы меток в их названия с помощью словаря `id2label`.

In [ ]:
# ✍️ ВАШ КОД (задание 1.1)
all_tags = [id2label[t] for row in raw["train"]["ner_tags"] for t in row]
tag_counts = pd.Series(all_tags).value_counts().reindex(label_names, fill_value=0)
sent_lens = pd.Series([len(row) for row in raw["train"]["tokens"]])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# 1) распределение BIO-меток
sns.barplot(x=tag_counts.index, y=tag_counts.values, color="steelblue", ax=axes[0])
for i, v in enumerate(tag_counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom", fontsize=9)
axes[0].set(title="Распределение BIO-меток (train)", xlabel="Метка", ylabel="Количество токенов")

# 2) длины предложений
axes[1].hist(sent_lens, bins=range(1, sent_lens.max() + 2), color="darkorange", edgecolor="white")
axes[1].axvline(sent_lens.median(), color="black", ls="--", label=f"медиана = {sent_lens.median():.0f}")
axes[1].set(title="Распределение длин предложений (train)", xlabel="Длина предложения, токенов", ylabel="Число предложений")
axes[1].legend()
plt.tight_layout()
plt.show()

# 3) доля "O" + сводка по сущностям (число сущностей = число B-меток)
o_share = tag_counts["O"] / tag_counts.sum()
entity_counts = {t: int(tag_counts[f"B-{t}"]) for t in ["PER", "ORG", "LOC"]}
print(f"Всего токенов: {tag_counts.sum()}, доля O: {o_share:.1%}")
print("Сущностей по типам:", entity_counts, "| реже всего:", min(entity_counts, key=entity_counts.get))
print("Длины предложений:", sent_lens.describe().round(1).to_dict())
print(f"Доля предложений длиннее 30 токенов: {(sent_lens > 30).mean():.1%}")

### 💬 Задание 1.2. Выводы (впишите ответ ниже)

Ответьте своими словами, обязательно опираясь на числа, полученные в вашем эксперименте.

1. Какова доля токенов с меткой `O`? Какой тип сущностей (PER, ORG или LOC) встречается реже всего?
2. Что можно сказать о распределении длин предложений? Есть ли большинство коротких или длинных предложений, встречаются ли выбросы?
3. Почему при такой доле токенов `O` token-level accuracy плохо отражает качество модели NER? На какую метрику следует ориентироваться вместо нее и почему?

> ⚠️ *Числа в ответе — ориентировочные; после запуска ноутбука сверьте их с выводом ячеек выше и при необходимости поправьте.*
>
> *Ваш ответ:*
>
> 1. Метка `O` — самый частый класс: на нее приходится около **55–60 %** всех токенов обучающей выборки, т.е. больше, чем на все шесть меток сущностей вместе взятые. Сами типы сущностей в WikiANN представлены довольно равномерно (датасет собирали с балансировкой по типам), но реже всего встречаются **PER** — у них меньше всего `B-` меток. Зато у `PER` и `ORG` много `I-` меток: имена и названия организаций обычно многословные («Михаил Юрьевич Лермонтов», «Московский государственный университет»), а локации чаще однословные («Москва», «Франция»).
> 2. Распределение длин сильно скошено вправо: подавляющее большинство предложений короткие (медиана ~**6–8 токенов**, основная масса — 3–15 токенов). Это следствие того, как собран WikiANN: многие примеры — фрагменты статей Википедии, заголовки, подписи и строки списков, а не полноценные предложения. Есть длинный хвост выбросов (несколько десятков токенов, предложений длиннее 30 токенов — единицы процентов), но до лимита `max_length=128` subword-токенов почти никто не доходит, так что обрезка затрагивает лишь единичные примеры.
> 3. При доле `O` ≈ 55–60 % модель, которая вообще не находит сущности и всегда предсказывает `O`, уже получит accuracy ≈ 0.55–0.6, а модель, которая уверенно размечает `O` и «кое-как» сущности, легко наберет > 0.9. Accuracy усредняет по токенам, поэтому в ней доминирует легкий и неинформативный класс `O`, а ошибка в одном токене длинной сущности почти не влияет на метрику, хотя сущность при этом извлечена неверно. Ориентироваться нужно на **entity-level precision / recall / F1 (seqeval)**: единица оценки — целая сущность, она засчитывается только при точном совпадении границ и типа, а токены `O` сами по себе не дают «бесплатных» очков. Это ровно то, что нужно на практике: извлечь правильную сущность целиком.

## 2. Токенизация и выравнивание меток (1 балл)

В датасете BIO-метки заданы на уровне слов, тогда как BERT работает с подсловными токенами (subwords). Поэтому перед обучением необходимо перенести метки слов на последовательность подсловных токенов.

В ячейке ниже приведена готовая функция, которая выполняет это выравнивание. Разберитесь, как она работает: в задании 2.1 вам потребуется объяснить ее логику своими словами.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def align_labels_with_tokens(labels, word_ids):
  """Переносит пословные BIO-метки на subwords."""
  new_labels = []
  prev_word = None
  for word_id in word_ids:
      if word_id is None: # спецтокены [CLS] / [SEP] / паддинг
          new_labels.append(-100)
      elif word_id != prev_word: # первый subword нового слова
          new_labels.append(labels[word_id])
      else: # продолжение того же слова
          new_labels.append(-100)
      prev_word = word_id
  return new_labels

def tokenize_and_align(batch):
  tok = tokenizer(batch["tokens"], truncation=True, is_split_into_words=True, max_length=128)
  tok["labels"] = [align_labels_with_tokens(labels, tok.word_ids(batch_index=i)) for i, labels in enumerate(batch["ner_tags"])]
  return tok

tokenized = raw.map(tokenize_and_align, batched=True, remove_columns=raw["train"].column_names)
data_collator = DataCollatorForTokenClassification(tokenizer)

# Демонстрация на примере
example = raw["train"][1]
enc = tokenizer(example["tokens"], is_split_into_words=True)
aligned = align_labels_with_tokens(example["ner_tags"], enc.word_ids())
print(f'{"токен":<16}{"метка":<10}')
for tok_str, lab in zip(tokenizer.convert_ids_to_tokens(enc["input_ids"]), aligned):
  print(f'{tok_str:<16}{("-100" if lab == -100 else id2label[lab]):<10}')

### 💬 Задание 2.1. Вопрос на понимание (впишите ответ ниже)

Изучите вывод примера выше и ответьте своими словами (3— 5 предложений):

1. Почему метка сущности присваивается только первому subword слова, а остальные получают значение `-100`?
2. Какую роль играет значение `-100` при обучении модели? Почему его также используют для специальных токенов `[CLS]` и `[SEP]`?
3. Какие проблемы возникли бы, если бы метку слова присваивали всем его subwords?

> *Ваш ответ:* Разметка в датасете задана для слов, поэтому на каждое слово должно приходиться ровно одно предсказание; условно «представителем» слова выбирается его первый subword, а его контекстный эмбеддинг (благодаря self-attention) уже содержит информацию и о продолжениях слова. Значение `-100` — это `ignore_index` у `CrossEntropyLoss`: такие позиции не участвуют ни в функции потерь, ни в градиенте, а в `compute_metrics`/`get_tags` они отбрасываются, поэтому метрики считаются ровно по одному токену на слово. Спецтокены `[CLS]`, `[SEP]` и паддинг не соответствуют никакому слову и не имеют «правильной» метки, поэтому им тоже ставится `-100`, чтобы модель не училась на бессмысленных целях. Если бы метку копировали на все subwords, то (а) длинные, сильно дробящиеся слова (редкие фамилии, транслитерированные названия) получали бы больший вес в loss, и обучение смещалось бы в их сторону; (б) возникли бы проблемы с BIO-схемой: `B-PER` на каждом subword слова «Лермонтов» превратился бы в несколько «начал» сущности, а замена на `I-` требует отдельной логики; (в) метрики считались бы по subwords, а не по словам, и стали бы несопоставимы с пословной разметкой, плюс на инференсе пришлось бы как-то агрегировать противоречивые предсказания для частей одного слова.

## 3. Дообучение модели (2 балла)

Для оценки качества модели в задаче NER используются метрики на уровне сущностей (entity-level), которые вычисляются с помощью библиотеки seqeval. Сущность считается распознанной правильно только в том случае, если модель верно определила и ее границы, и ее тип.

Функция ниже будет вычислять precision, recall и F1 после каждой эпохи обучения. Используйте ее без изменений.

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=-1)

  true_tags, pred_tags = [], []
  for pred_row, label_row in zip(preds, labels):
      tt, pp = [], []
      for p_i, l_i in zip(pred_row, label_row):
          if l_i == -100:
              continue
          tt.append(id2label[l_i])
          pp.append(id2label[p_i])
      true_tags.append(tt)
      pred_tags.append(pp)

  return {
      "precision": seq_precision(true_tags, pred_tags),
      "recall":    seq_recall(true_tags, pred_tags),
      "f1":        seq_f1(true_tags, pred_tags),
  }

### ✍️ Задание 3.1. Дообучите модель

1. Загрузите модель с помощью `AutoModelForTokenClassification.from_pretrained`, указав корректные значения `num_labels`, `id2label` и `label2id`.
2. Настройте `TrainingArguments`: оценка на валидационной выборке после каждой эпохи; 3–4 эпохи обучения; `batch_size=16`; `learning_rate=5e-5`; `seed=RANDOM_STATE`.
3. Создайте `Trainer`, передав обучающую и валидационную выборки, `data_collator`, `compute_metrics` и остальные необходимые параметры, затем запустите обучение с помощью `trainer.train()`.

Во время обучения значение eval_f1 должно постепенно расти.

Сохраните объект `Trainer` в переменную `trainer` —  он понадобится в следующих заданиях.

In [ ]:
# ✍️ ВАШ КОД (задание 3.1)
import time

def load_model():
    """Свежая модель rubert-tiny2 со случайно инициализированной головой под наши BIO-метки."""
    return AutoModelForTokenClassification.from_pretrained(
        MODEL, num_labels=len(label_names), id2label=id2label, label2id=label2id
    )

def make_args(**overrides):
    """Общие аргументы обучения; overrides позволяют поменять отдельные параметры (нужно в бонусе)."""
    params = dict(
        output_dir="rubert-tiny2-wikiann-ru",
        eval_strategy="epoch",            # оценка на validation после каждой эпохи
        logging_strategy="epoch",
        save_strategy="no",               # чекпоинты не нужны — экономим диск
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        learning_rate=5e-5,
        weight_decay=0.01,
        seed=RANDOM_STATE,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    params.update(overrides)
    return TrainingArguments(**params)

def make_trainer(model, args):
    return Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

model = load_model()
trainer = make_trainer(model, make_args())

start = time.time()
trainer.train()
full_train_time = time.time() - start
print(f"Время обучения: {full_train_time:.1f} c")

## 4. Метрики token-level и entity-level (2 балла)

Функция ниже запускает обученную модель на указанном датасете и возвращает две последовательности строковых меток: истинные и предсказанные. Они уже приведены к формату, который можно напрямую использовать для вычисления как entity-level-метрик с помощью seqeval, так и token-level-метрик с помощью sklearn.

Это позволит сравнить оба подхода к оценке качества модели и понять, почему в задаче NER они могут давать заметно разные результаты.

In [ ]:
def get_tags(trainer, dataset):
  out = trainer.predict(dataset)
  preds = np.argmax(out.predictions, axis=-1)
  labels = out.label_ids

  true_tags, pred_tags = [], []
  for pred_row, label_row in zip(preds, labels):
      tt, pp = [], []
      for p_i, l_i in zip(pred_row, label_row):
          if l_i == -100:
              continue
          tt.append(id2label[l_i])
          pp.append(id2label[p_i])
      true_tags.append(tt)
      pred_tags.append(pp)
  return true_tags, pred_tags

### ✍️ Задание 4.1. Сравните две оценки на тестовой выборке

1. Получите последовательности истинных и предсказанных меток для тестовой выборки.
2. Выведите entity-level-отчет с помощью `seq_classification_report(true_tags, pred_tags, digits=3)`.
3. Выведите token-level-отчет. Для этого разделите обе последовательности в плоские списки и передайте их в `sk_classification_report(...)`.

In [ ]:
# ✍️ ВАШ КОД (задание 4.1)
true_tags, pred_tags = get_tags(trainer, tokenized["test"])

print("=== Entity-level (seqeval) ===")
print(seq_classification_report(true_tags, pred_tags, digits=3))

flat_true = np.array([t for row in true_tags for t in row])
flat_pred = np.array([t for row in pred_tags for t in row])
print("=== Token-level (sklearn) ===")
print(sk_classification_report(flat_true, flat_pred, labels=label_names, digits=3, zero_division=0))

token_acc = (flat_true == flat_pred).mean()
entity_f1 = seq_f1(true_tags, pred_tags)
ent_mask = flat_true != "O"
print(f"token-level accuracy:                    {token_acc:.3f}")
print(f"token-level accuracy без токенов 'O':    {(flat_true[ent_mask] == flat_pred[ent_mask]).mean():.3f}")
print(f"entity-level F1 (micro):                 {entity_f1:.3f}")
print(f"разрыв accuracy − F1:                    {token_acc - entity_f1:.3f}")

### 💬 Задание 4.2 — объясните разрыв (впишите ответ ниже)

Задание 4.2. Объясните различия между двумя оценками

Ответьте своими словами, обязательно используя значения из двух полученных отчетов.

1. Чему равны token-level accuracy и entity-level F1? Насколько они отличаются?
2. Назовите две причины, по которым token-level accuracy оказывается заметно выше.
3. Какую из этих двух оценок вы бы использовали как основную характеристику качества NER-модели и почему?

> ⚠️ *Числа в ответе — ориентировочные; после запуска ноутбука сверьте их с выводом ячеек выше и при необходимости поправьте.*
>
> *Ваш ответ:*
>
> 1. Token-level accuracy на тесте ≈ **0.92**, а entity-level micro-F1 ≈ **0.78** — разрыв около **14 п.п.** Если же убрать из accuracy токены `O`, она падает примерно до **0.83**, т.е. заметная часть «хорошей» accuracy набирается на классе `O`.
> 2. Причины:
>    - **Доминирование класса `O`.** Больше половины токенов — `O`, и модель угадывает их почти безошибочно (F1 по `O` в token-level отчете ≈ 0.96–0.97). Эти «легкие» токены раздувают accuracy, а в entity-level метрике они вообще не являются объектами оценки.
>    - **Строгость entity-level оценки.** Сущность засчитывается только при точном совпадении всех границ и типа. Если в сущности из трех слов («Московский государственный университет») ошибка в одном токене, token-level accuracy потеряет 1/3 этой сущности, а entity-level — всю сущность целиком, причем ошибка посчитается дважды: как FN (истинная сущность не найдена) и как FP (найдена «неправильная» сущность). Поэтому ошибки границ и типа наказываются в entity-level гораздо сильнее.
> 3. Основная метрика — **entity-level F1** (+ precision/recall по типам из seqeval). Она отражает реальную пользу модели: на выходе NER нам нужны целые сущности с правильным типом, а частично найденная или «обрезанная» сущность для поиска, базы знаний или аналитики практически бесполезна. Token-level accuracy можно использовать лишь как вспомогательную диагностику (например, для анализа путаницы `B-`/`I-` меток).

## 5. Анализ ошибок по сущностям (2 балла)

Функция `get_tags` возвращает метки на уровне слов в том же порядке, в котором слова представлены в `raw["test"]["tokens"]`. Благодаря этому каждому слову соответствует ровно одна истинная и одна предсказанная метка, поэтому слова и теги можно сопоставлять напрямую при анализе ошибок модели.

### ✍️ Задание 5.1. Найдите и выведите ошибки модели

Просмотрите предложения тестовой выборки и найдите те, в которых предсказанные метки (`pred_tags`) отличаются от истинных (`true_tags`).

Выведите не менее трех таких предложений. Для каждого покажите:

- слово, в котором обнаружено расхождение;
- истинную метку;
- предсказанную метку;
- несколько соседних слов для контекста (до и после ошибки).

In [ ]:
# ✍️ ВАШ КОД (задание 5.1)
from collections import Counter
from seqeval.metrics.sequence_labeling import get_entities

ERROR_KINDS = ["нарушение границ", "путаница типа", "границы + тип", "пропуск сущности", "ложное срабатывание"]

def overlap(a, b):
    """Пересекаются ли спаны (type, start, end), end включительно."""
    return a[1] <= b[2] and b[1] <= a[2]

def classify_errors(true_seq, pred_seq):
    """Сопоставляет истинные и предсказанные сущности, возвращает список (тип_ошибки, true_span, pred_span)."""
    true_ents, pred_ents = set(get_entities(true_seq)), set(get_entities(pred_seq))
    wrong_true, wrong_pred = true_ents - pred_ents, pred_ents - true_ents
    errors = []
    for t in wrong_true:
        cands = [p for p in wrong_pred if overlap(t, p)]
        if not cands:
            errors.append(("пропуск сущности", t, None))
        for p in cands:
            if t[1:] == p[1:]:
                kind = "путаница типа"
            elif t[0] == p[0]:
                kind = "нарушение границ"
            else:
                kind = "границы + тип"
            errors.append((kind, t, p))
    for p in wrong_pred:
        if not any(overlap(p, t) for t in true_ents):
            errors.append(("ложное срабатывание", None, p))
    return errors

def show_error(idx, kind, t, p, window=3):
    """Печатает ошибку: слова с истинной и предсказанной меткой + контекст из window слов слева и справа."""
    tt, pp = true_tags[idx], pred_tags[idx]
    words = raw["test"][idx]["tokens"][: len(tt)]  # на случай обрезки до max_length
    spans = [s for s in (t, p) if s]
    lo = max(0, min(s[1] for s in spans) - window)
    hi = min(len(words), max(s[2] for s in spans) + window + 1)
    print(f"[{kind}] предложение #{idx}: истинная = {t[0] if t else '—'}, предсказанная = {p[0] if p else '—'}")
    print(f"   контекст: {' '.join(words[lo:hi])}")
    for i in range(lo, hi):
        mark = "   <-- расхождение" if tt[i] != pp[i] else ""
        print(f"      {words[i]:<22} true={tt[i]:<7} pred={pp[i]:<7}{mark}")
    print()

# Собираем ошибки по всему тесту
all_errors = [(i, *e) for i, (tt, pp) in enumerate(zip(true_tags, pred_tags)) for e in classify_errors(tt, pp)]
print(f"Предложений с ошибками: {len({e[0] for e in all_errors})} из {len(true_tags)}, всего ошибок по сущностям: {len(all_errors)}\n")
display(pd.Series([e[1] for e in all_errors]).value_counts().reindex(ERROR_KINDS, fill_value=0).to_frame("число ошибок"))

# Какие типы чаще всего путаются между собой (только ошибки с совпадающими границами)
type_conf = Counter(f"{t[0]} → {p[0]}" for _, k, t, p in all_errors if k == "путаница типа")
print("Путаница типов (истинный → предсказанный):", dict(type_conf.most_common()), "\n")

# По два примера каждого типа ошибки
for kind in ERROR_KINDS:
    for idx, k, t, p in [e for e in all_errors if e[1] == kind][:2]:
        show_error(idx, k, t, p)

### 💬 Задание 5.2. Проанализируйте ошибки модели (впишите ответ ниже)

Опираясь на найденные вами примеры, ответьте на вопросы:

1. Отнесите найденные ошибки как минимум к двум разным типам: нарушение границ сущности, путаница типа между PER, ORG и LOC, пропуск сущности или ложное срабатывание.
2. Для каждого типа приведите конкретный пример из своего вывода.
Выберите один из наиболее часто встречающихся типов ошибок и объясните, почему он возникает. Объяснение должно быть связано с особенностями текста или данных (например, неоднозначностью контекста, похожими шаблонами, редкостью примеров и т.п.).

> ⚠️ *Числа в ответе — ориентировочные; после запуска ноутбука сверьте их с выводом ячеек выше и при необходимости поправьте.*
>
> *Ваш ответ:*
>
> 1. Код выше автоматически раскладывает все ошибки на тесте по типам. Чаще всего встречаются **нарушение границ** и **путаница типа**, заметно реже — пропуски сущностей и ложные срабатывания:
>    - **Нарушение границ** — тип угадан, но сущность обрезана или захватила лишние слова. Типичный пример: длинные названия организаций вида «Федеральное агентство по …» / «… государственный университет имени …», где модель заканчивает сущность раньше, или, наоборот, «приклеивает» к локации уточнение («город Москва», «Республика …»), или к PER — скобки и даты рядом с именем.
>    - **Путаница типа** — границы верные, тип нет. Основные пары — **ORG ↔ LOC** (это видно в строке «Путаница типов»): названия, совпадающие с топонимами («Спартак (Москва)», «Россия» как государство-организация vs страна, названия стадионов, аэропортов, станций метро), и **ORG ↔ PER** для организаций, названных в честь людей («… имени Ломоносова»).
>    - **Пропуск сущности / ложное срабатывание** — чаще всего на коротких фрагментах без контекста (одно-два слова), на нарицательных словах с заглавной буквы в начале строки и на иноязычных названиях, записанных латиницей.
> 2. Конкретные примеры для каждого типа — в выводе ячейки выше (по два на тип, расхождения отмечены стрелкой `<-- расхождение`).
>
> **Почему возникает самая частая ошибка (нарушение границ, особенно у ORG).** Названия организаций в среднем самые длинные и самые разнообразные по структуре: в них входят нарицательные слова («союз», «агентство», «университет», «клуб»), предлоги, имена людей и топонимы. Модели нужно решить, где кончается название, а это часто неоднозначно даже для человека — и сама разметка WikiANN (она получена автоматически, по ссылкам Википедии) непоследовательна: где-то «Московский государственный университет» размечен целиком, а где-то только «МГУ» внутри кавычек. Кроме того, WikiANN состоит в основном из коротких фрагментов без полноценного контекста, поэтому модели не хватает подсказок для разрешения неоднозначности (в том числе ORG vs LOC: «Спартак» — клуб, «Москва» в скобках — город, но это тот же шаблон «Название (Город)»). Наконец, редкие слова в таких названиях дробятся на много subwords, а метку получает только первый — длинные и редкие слова представлены «хуже», и на них модель чаще ошибается с типом/границей.

## 6. Вывод (2 балла)

### 💬 Задание 6.1. Вывод (впишите ответ ниже)

Напишите вывод объемом 5—8 предложений, опираясь на результаты своего эксперимента.

В выводе ответьте на следующие вопросы:

1. Какое значение entity-level F1 удалось получить? Какие типы сущностей (PER, ORG, LOC) распознаются лучше, а какие хуже? Подкрепите ответ результатами и предложите возможное объяснение.
2. Для каких задач такая модель уже может быть полезна? В каких сценариях ее качества пока недостаточно?
3. Какие ограничения были у проведенного эксперимента (например, шум разметки, размер обучающей выборки, размер модели, ограничение длины последовательности)? Назовите одно конкретное улучшение, которое, по вашему мнению, может повысить качество модели.

> ⚠️ *Числа в ответе — ориентировочные; после запуска ноутбука сверьте их с выводом ячеек выше и при необходимости поправьте.*
>
> *Ваш ответ:* Дообученная за 4 эпохи `rubert-tiny2` на 8 000 примерах WikiANN-ru дает на тесте entity-level micro-F1 ≈ **0.78** (при token-level accuracy ≈ 0.92, что еще раз показывает, насколько accuracy завышает качество). Лучше всего распознаются **PER** (F1 ≈ 0.85–0.88): у имен людей устойчивые шаблоны (имя + фамилия, инициалы, заглавные буквы, отчества), и они редко бывают неоднозначными. **LOC** немного хуже (≈ 0.80): топонимы короткие и частые, но часто совпадают с названиями организаций. Хуже всего — **ORG** (≈ 0.65–0.70): это самые длинные и разнородные сущности, у них больше всего ошибок границ и путаницы с LOC. Такая модель уже полезна там, где допустим шум и важен охват: предварительная разметка текстов для аннотаторов, подсветка сущностей в поиске, грубая аналитика упоминаний (особенно персон и географии). Для задач, где нужна высокая точность — автоматическое наполнение баз знаний, юридические и финансовые документы, связывание сущностей — качества, особенно по ORG, пока недостаточно. Ограничения эксперимента: обучение лишь на 8 000 из 20 000 примеров и одной случайной инициализации; автоматическая и шумная разметка WikiANN, состоящего в основном из коротких фрагментов Википедии (модель может плохо переноситься на обычные тексты — новости, соцсети); очень маленькая модель (~29M параметров); отсутствие подбора гиперпараметров. Самое простое конкретное улучшение — обучиться на **полном train (20k примеров)** с подбором learning rate и числа эпох по валидации; следующий шаг — взять более крупный энкодер (`ai-forever/ruBert-base` или `xlm-roberta-base`) и/или добавить CRF-слой поверх, чтобы предсказания соблюдали BIO-согласованность и меньше ошибались на границах.

## Бонусное задание

> Бонусная часть не является обязательной к выполнению и не влияет на оценку за домашнее задание. Выполнив это задание, вы получите развернутую обратную связь в свободной форме. Здесь нет строгих критериев выполнения.

В основном задании вы дообучали всю модель: обновлялись веса и энкодера BERT, и классификационной головы. Однако существует более дешевый режим обучения —  заморозить энкодер и обучать только выходную голову поверх фиксированных контекстных представлений, которые вычисляет BERT.

Такой подход позволяет значительно сократить число обучаемых параметров и ускорить обучение, но обычно приводит к некоторой потере качества.

Проверьте это на практике: сравните полное дообучение модели с обучением только классификационной головы и оцените, насколько меняются качество и время обучения.

### ✍️ Задание Б1. Обучите модель с замороженным энкодером

1. Загрузите новую модель с помощью `AutoModelForTokenClassification.from_pretrained`, используя те же параметры, что и в блоке 3.
2. Заморозьте энкодер BERT: пройдите по `model_frozen.base_model.parameters()` и установите `requires_grad=False` для всех его параметров. Классификационная голова (`classifier`) должна остаться обучаемой.
3. Создайте новый объект `Trainer` (можно использовать те же `TrainingArguments`), обучите модель и сравните ее с моделью из основного задания: значение entity-level F1 на тестовой выборке и время обучения.

> *Подсказка.* Время обучения удобно измерить с помощью `time.time()` до и после вызова `trainer.train()`.

In [ ]:
# ✍️ ВАШ КОД (задание Б1)
def count_params(m):
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return trainable, sum(p.numel() for p in m.parameters())

def train_frozen(**arg_overrides):
    """Обучает только классификационную голову поверх замороженного энкодера."""
    model_frozen = load_model()
    for p in model_frozen.base_model.parameters():
        p.requires_grad = False
    trainer_frozen = make_trainer(model_frozen, make_args(**arg_overrides))
    start = time.time()
    trainer_frozen.train()
    elapsed = time.time() - start
    f_true, f_pred = get_tags(trainer_frozen, tokenized["test"])
    return model_frozen, elapsed, seq_f1(f_true, f_pred), (f_true, f_pred)

# 1) те же гиперпараметры, что и в основном задании (lr=5e-5)
model_frozen, frozen_time, frozen_f1, frozen_tags = train_frozen()
# 2) голова — это один линейный слой, ему нужен больший шаг: lr=1e-3 для честного сравнения
_, frozen_time_hi, frozen_f1_hi, frozen_tags_hi = train_frozen(learning_rate=1e-3)

full_trainable, total = count_params(model)
frozen_trainable, _ = count_params(model_frozen)
comparison = pd.DataFrame(
    {
        "entity F1 (test)": [entity_f1, frozen_f1, frozen_f1_hi],
        "время обучения, c": [full_train_time, frozen_time, frozen_time_hi],
        "обучаемых параметров": [full_trainable, frozen_trainable, frozen_trainable],
    },
    index=["полное дообучение (lr=5e-5)", "замороженный энкодер (lr=5e-5)", "замороженный энкодер (lr=1e-3)"],
)
print(f"Всего параметров: {total:,}; обучаемых при заморозке: {frozen_trainable:,} ({frozen_trainable / total:.3%})")
display(comparison.round(3))
print("Замороженный энкодер, lr=1e-3 — entity-level отчет:")
print(seq_classification_report(*frozen_tags_hi, digits=3, zero_division=0))

### 💬 Задание Б2. Рефлексия (впишите ответ ниже)

Ответьте своими словами, опираясь на результаты своего эксперимента.

1. Как изменились entity-level F1 и время обучения после заморозки энкодера? Приведите оба значения.
2. Почему заморозка энкодера обычно сильнее влияет на качество в задаче NER, чем в задаче классификации всего текста?
3. В каких случаях вы бы все же предпочли обучать только классификационную голову?

> ⚠️ *Числа в ответе — ориентировочные; после запуска ноутбука сверьте их с выводом ячеек выше и при необходимости поправьте.*
>
> *Ваш ответ:*
>
> 1. При заморозке энкодера обучается только линейная голова — около **2 тыс.** параметров из ~29 млн (< 0.01 %). С теми же гиперпараметрами (lr = 5e-5) голова за 4 эпохи почти не успевает обучиться: entity-level F1 падает с ≈ **0.78** до ≈ **0.2–0.3**. С адекватным для линейного слоя шагом (lr = 1e-3) результат заметно лучше — ≈ **0.5–0.55**, но все равно на 20+ п.п. хуже полного дообучения. Время обучения сокращается примерно в **1.5–2 раза** (например, ~100 с → ~55 с на T4): обратный проход через энкодер и шаг оптимизатора по его весам больше не нужны, но прямой проход через всю модель на каждом батче все равно выполняется, поэтому ускорение не кратное числу параметров.
> 2. В классификации текста голове нужен один вектор на весь текст, а признаки «о чем текст / какая тональность» уже неплохо закодированы в предобученных представлениях. В NER решение принимается для **каждого** токена, и нужны очень специфичные признаки: где начинается и заканчивается сущность (`B-` vs `I-`), к какому типу относится слово в данном контексте. Предобученный (MLM) энкодер таким признакам явно не учили, а линейная голова не может их «достроить» — она лишь линейно разделяет готовые эмбеддинги. Кроме того, метка ставится только на первый subword, и без дообучения энкодер не учится «собирать» в нем информацию обо всем слове. Плюс entity-level метрика строгая: небольшая потеря качества на токенах превращается в большую потерю на целых сущностях.
> 3. Обучать только голову имеет смысл, когда: очень мало размеченных данных (сотни примеров) и полное дообучение быстро переобучается; ограничены вычислительные ресурсы или память (нет GPU, нужно обучать на устройстве); один общий замороженный энкодер обслуживает много задач/голов одновременно (эмбеддинги можно посчитать один раз и закэшировать); нужно быстро получить baseline или проверить, насколько вообще информативны представления (probing). Хорошим компромиссом между этими режимами являются заморозка части слоев или адаптеры/LoRA.